<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 1 (Lunes) — Predicción — Series de tiempo**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# ⏳ Series de tiempo: ARIMA construido desde cero

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Entender qué es una serie de tiempo, sus componentes y el modelo ARIMA(p, d, q). |
| 🔍 Comprensión de los datos | Cargamos `df.csv` (transacciones), agregamos por mes y descomponemos la serie. |
| 🧹 Preparación de los datos | Hacemos la serie estacionaria (logaritmo, diferenciación) y verificamos con la prueba ADF. |
| 🤖 Modelado | Implementamos AR y MA desde cero y luego usamos `auto_arima` y `ARIMA` de `statsmodels`. |
| 📊 Evaluación | Partimos en entrenamiento/prueba **cronológicamente** y medimos el RMSE. |
| 🚀 Despliegue y conclusiones | Comparamos valores reales vs. pronóstico y proponemos cómo llevarlo a producción. |

> 🎯 **Analogía:** ARIMA es una receta con tres ingredientes: **AR** (el pasado de la serie), **I** (diferenciar para quitar la tendencia) y **MA** (los errores pasados); aquí los preparamos a mano antes de usar la librería.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 👉 **1** | Lunes | **Predicción: regresión lineal y series de tiempo** |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('df.csv', 'dia_2')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Entender qué es una serie de tiempo, sus componentes y el modelo ARIMA(p, d, q).

📚 Lectura recomendada: [3] [4]

# **Series de tiempo**

Es una sucesión de datos medidos en determinados momentos y ordenados cronológicamente. Los datos pueden estar espaciados a intervalos iguales (como la temperatura en un observatorio meteorológico en días sucesivos al mediodía) o desiguales (como el peso de una persona en sucesivas mediciones en el consultorio médico, la farmacia, etc.). Para el análisis de las series temporales se usan métodos que ayudan a interpretarlas y que permiten extraer información representativa sobre las relaciones subyacentes entre los datos de la serie o de diversas series y que permiten en diferente medida y con distinta confianza extrapolar o interpolar los datos y así predecir el comportamiento de la serie en momentos no observados, sean en el futuro (extrapolación pronóstica), en el pasado (extrapolación retrógrada) o en momentos intermedios (interpolación)



Cualquier valor de una serie es el producto de factores que se pueden
atribuir a las cuatro componentes.

**1. Tendencia secular**: La tendencia secular o tendencia a largo plazo de una serie es por lo común el resultado de factores a largo plazo. En términos intuitivos, la tendencia de una serie de tiempo caracteriza el patrón gradual y consistente de las variaciones de la propia serie, que se consideran consecuencias de fuerzas persistentes que afectan el crecimiento o la reducción de la misma, tales como:
cambios en la población, en las características demográficas de la misma, cambios en los ingresos, en la salud, en el nivel de educación y tecnología. Las tendencias a largo plazo se ajustan a diversos esquemas. Algunas se mueven continuamente hacía arriba, otras declinan, y otras más permanecen igual en un cierto período o intervalo de tiempo.




**2. Variación estacional**: El componente de la serie de tiempo que representa la variabilidad en los datos debida a influencias de las estaciones, se llama
componente estacional. Esta variación corresponde a los movimientos de la serie
que recurren año tras año en los mismos meses (o en los mismos trimestres) del año poco más o menos con la misma intensidad. Por ejemplo: Un fabricante de albercas inflables espera poca actividad de ventas durante los meses de otoño e invierno y tiene ventas máximas en los de primavera y verano, mientras que los fabricantes de equipo para la nieve y ropa de abrigo esperan un comportamiento anual opuesto al del fabricante de albercas.


**3. Variación cíclica**: Con frecuencia las series de tiempo presentan secuencias alternas de puntos abajo y arriba de la línea de tendencia que duran más de un año, esta variación se mantiene después de que se han eliminado las variaciones o tendencias estacional e irregular. Un ejemplo de este tipo de variación son los ciclos comerciales cuyos períodos recurrentes dependen de la prosperidad, recesión, depresión y recuperación, las cuales no dependen de factores como el clima o las costumbres sociales.


**4. Variación Irregular**: Esta se debe a factores a corto plazo, imprevisibles y no recurrentes que afectan a la serie de tiempo. Como este componente explica la variabilidad aleatoria de la serie, es impredecible, es decir, no se puede esperar predecir su impacto sobre la serie de tiempo. Existen dos tipos de variación irregular:
a) Las variaciones que son provocadas por acontecimientos especiales, fácilmente identificables, como las elecciones, inundaciones, huelgas, terremotos.
b)Variaciones aleatorias o por casualidad, cuyas causas no se pueden señalar en forma exacta, pero que tienden a equilibrarse a la larga.

![Serie de tiempo real (manchas solares) con su ACF e histograma](https://raw.githubusercontent.com/alkaline-ml/pmdarima/master/doc/img/sunspots/untransformed.png)
*Figura: Serie de tiempo real (manchas solares) con su ACF e histograma. [8]*

**Autoregressive Integrated Moving Average Model**

Un modelo **ARIMA** es una clase de modelos estadísticos para analizar y pronosticar datos de series de tiempo.

Atiende explícitamente a un conjunto de estructuras estándar en datos de series de tiempo y, como tal, proporciona un método simple pero poderoso para realizar pronósticos de series de tiempo hábiles.

ARIMA es un acrónimo de AutoRegressive Integrated Moving Average. Es una generalización de **la media móvil autorregresiva más simple y agrega la noción de integración.**

*   **Parámetro AR (Modelo Autorregresivo)**: Este parámetro nos dice cómo los valores anteriores de la cosa que estamos analizando afectan los valores futuros. Imagina que estamos hablando del clima. El parámetro AR nos dice cuánto influye la temperatura de ayer en la temperatura de hoy. Si el parámetro AR es alto, significa que el clima de hoy está muy relacionado con el clima de ayer. Si el parámetro AR es bajo, entonces el clima de hoy no está muy relacionado con el clima de ayer.

*   **Parámetro I (Integración)**: Este parámetro nos dice cuántas veces tenemos que diferenciar la serie de tiempo para que sea estacionaria. Si la serie de tiempo no es estacionaria, significa que hay algún patrón o tendencia que cambia con el tiempo y que hace que sea difícil predecir el comportamiento futuro. La integración nos ayuda a hacer que la serie de tiempo sea estacionaria, lo que facilita las predicciones futuras.

*   **Parámetro MA (Modelo de Media Móvil)**: Este parámetro nos dice cómo los errores anteriores de la cosa que estamos analizando afectan los errores futuros. Siguiendo con el ejemplo del clima, el parámetro MA nos dice cuánto influye el error que cometimos al predecir la temperatura de ayer en el error que cometimos al predecir la temperatura de hoy. Si el parámetro MA es alto, significa que los errores anteriores influyen mucho en los errores futuros. Si el parámetro MA es bajo, entonces los errores anteriores no influyen tanto en los errores futuros.

Cada uno de estos componentes se especifica explícitamente en el modelo como parámetro. Se utiliza una notación estándar de **ARIMA (p, d, q)** donde los parámetros se sustituyen por valores enteros para indicar rápidamente el modelo ARIMA específico que se está utilizando.

Los parámetros del modelo ARIMA se definen de la siguiente manera:

**p:** El número de observaciones de retraso incluidas en el modelo, también llamado orden de retraso.

**d:** El número de veces que se diferencian las observaciones sin procesar, también llamado grado de diferenciación.

**q:** El tamaño de la ventana de media móvil, también llamado orden de media móvil.

$$$$


**Criterio de información de Akaike / (AIC)**

El criterio de información de Akaike es una medida de la bondad de ajuste de un modelo estadístico. Se puede decir que describe la relación entre el sesgo y varianza en la construcción del modelo, o hablando de menera general acerca de la exactitud y complejidad del modelo.

El AIC no es una prueba del modelo en el sentido de prueba de hipótesis.Más bien, proporciona un medio para la comparación entre los modelos de una herramienta para la selección del modelo. Dado un conjunto de datos, varios modelos candidatos pueden ser clasificados de acuerdo a su AIC, con el modelo que tiene el **mínimo AIC es la mejor**. A partir de los valores de la AIC también se puede inferir que, por ejemplo, los dos primeros modelos estan más o menos empatados y el resto son mucho peores.

**NOTA**

En general, un modelo con un AIC menor se considera que se ajusta mejor a los datos y es más preciso. Por lo tanto, el AIC se utiliza a menudo como medida para comparar diferentes modelos ARIMA y elegir el modelo que mejor se ajuste a los datos.

Es importante tener en cuenta que el AIC no mide directamente la precisión de las predicciones del modelo, sino que se utiliza como una medida de selección de modelo para elegir entre varios modelos con diferentes parámetros. Por lo tanto, se recomienda utilizar medidas de error como el MSE o el MAE para evaluar la precisión de las predicciones de un modelo ARIMA específico.


In [ ]:
%pip install -q "pmdarima>=2.1.1"

In [ ]:
import csv
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore") #suppress warnings


import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

from pmdarima.arima import auto_arima

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.seasonal import seasonal_decompose
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
pd.options.mode.chained_assignment = None  # default='warn'


---
## 🤖 Fase 4 · Modelado — funciones AR, MA y prueba de estacionariedad ADF

Implementamos AR y MA desde cero y luego usamos `auto_arima` y `ARIMA` de `statsmodels`.

📚 Lectura recomendada: [4] [8]

In [ ]:
def AR(p, df):
    # Generating the lagged p terms
    for i in range(1, p+1):
        df[f'shifted_values_{i}'] = df['amount'].shift(i)

    train_size = int(0.8 * len(df))

    # Breaking data set into test and training
    df_train = df.iloc[:train_size].copy()
    df_test = df.iloc[train_size:].copy()

    # Drop missing values
    df_train.dropna(inplace=True)

    # X contains the lagged values , hence we skip the first column
    X_train = df_train.iloc[:, 1:].values.reshape(-1, p)
    # Y contains the value, it is the first column
    y_train = df_train.iloc[:, 0].values.reshape(-1, 1)

    # Running linear regression to generate the coefficients of lagged terms
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    theta = lr.coef_.T
    intercept = lr.intercept_

    df_train['Predicted_Values'] = X_train.dot(lr.coef_.T) + lr.intercept_
    # df_train[['Value','Predicted_Values']].plot()

    X_test = df_test.iloc[:, 1:].values.reshape(-1, p)
    df_test['Predicted_Values'] = X_test.dot(lr.coef_.T) + lr.intercept_
    # df_test[['Value','Predicted_Values']].plot()

    RMSE = np.sqrt(mean_squared_error(df_test['amount'], df_test['Predicted_Values']))

    print(f"The RMSE is: {RMSE}, Value of p: {p}")
    return [df_train, df_test, theta, intercept, RMSE]

In [ ]:
def MA(q, res):
    for i in range(1, q+1):
        res[f'Shifted_values_{i}'] = res['Residuals'].shift(i)

    train_size = int(0.8 * res.shape[0])

    res_train = res.iloc[:train_size]
    res_test = res.iloc[train_size:]

    res_train_2 = res_train.dropna()
    X_train = res_train_2.iloc[:, 1:].values.reshape(-1, q)
    y_train = res_train_2.iloc[:, 0].values.reshape(-1, 1)

    X_train = sm.add_constant(X_train)
    model = sm.OLS(y_train, X_train)
    results = model.fit()

    theta = results.params[1:].T
    intercept = results.params[0]
    res_train_2['Predicted_Values'] = results.fittedvalues
    # res_train_2[['Residuals', 'Predicted_Values']].plot()

    X_test = res_test.iloc[:, 1:].values.reshape(-1, q)
    X_test = sm.add_constant(X_test)
    res_test['Predicted_Values'] = results.predict(X_test)
    res_test[['Residuals', 'Predicted_Values']].plot()

    RMSE = np.sqrt(results.mse_resid)
    print(f"The RMSE is: {RMSE}, Value of q: {q}")
    return [res_train_2, res_test, theta, intercept, RMSE]

La función **adf_check()** toma como entrada una serie de tiempo y realiza una prueba de Dickey-Fuller aumentada (ADF). La prueba ADF se utiliza para determinar si una serie de tiempo es estacionaria o no. La función imprime el resultado de la prueba, que incluye el estadístico de prueba ADF, el valor p, el número de retardos utilizados en la prueba y el número de observaciones utilizadas. Luego, se imprime un mensaje que indica si hay suficiente evidencia para rechazar la hipótesis nula de que la serie de tiempo tiene una raíz unitaria y es no estacionaria, o si hay evidencia insuficiente para rechazar la hipótesis nula. Si el valor p es menor o igual a 0,05, hay suficiente evidencia para rechazar la hipótesis nula y concluir que la serie de tiempo es estacionaria. Si el valor p es mayor que 0,05, no hay suficiente evidencia para rechazar la hipótesis nula y se concluye que la serie de tiempo es no estacionaria.

In [ ]:
def adf_check(time_series):
    """
    Pasa una serie de tiempo y devuelve el informe ADF
    """
    result = adfuller(time_series)
    print('Prueba de Dickey-Fuller Aumentada:')
    labels = ['Estadístico de prueba ADF', 'Valor-p', 'Número de retrasos usados', 'Número de observaciones usadas']

    for value, label in zip(result, labels):
        print(label + ' : ' + str(value))

    if result[1] <= 0.05:
        print("Hay suficiente evidencia en contra de la hipótesis nula, rechazar la hipótesis nula. Los datos no tienen raíz unitaria y son estacionarios")
    else:
        print("Hay evidencia débil en contra de la hipótesis nula, la serie de tiempo tiene una raíz unitaria, lo que indica que no es estacionaria \n")


---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `df.csv` (transacciones), agregamos por mes y descomponemos la serie.

📚 Lectura recomendada: [6]

In [ ]:
#Leer los datos
df = pd.read_csv('sample_data/df.csv',parse_dates=True,index_col='date')
df

In [ ]:
dfGroupby=df.groupby(df.index.strftime('%Y-%m')).agg({'amount': 'sum'})

In [ ]:
dfGroupby

In [ ]:
index = pd.date_range(start=dfGroupby.index[0],periods=len(dfGroupby.index), freq='MS')
df1 = dfGroupby.reindex(index, fill_value=0)
df1["amount"]=dfGroupby["amount"].values

In [ ]:
df1

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df1.index, df1['amount'])
plt.title('Importe total por mes y año')
plt.xlabel('Fecha')
plt.ylabel('Monto')
plt.show()

In [ ]:
# Descomposición de la serie en componentes
decomposition = sm.tsa.seasonal_decompose(df1, model='additive')

# Graficar la tendencia
trend = decomposition.trend
trend.plot(figsize=(12,4), title='Tendencia')

# Graficar la estacionalidad
seasonality = decomposition.seasonal
seasonality.plot(figsize=(12,4), title='Estacionalidad')

In [ ]:
# Realizar descomposición
result = seasonal_decompose(df1['amount'], model='additive')

# Graficar componentes
fig, ax = plt.subplots(nrows=4, ncols=1, figsize=(10, 8))
ax[0].plot(result.observed)
ax[0].set_ylabel('Observado')
ax[1].plot(result.trend)
ax[1].set_ylabel('Tendencia')
ax[2].plot(result.seasonal)
ax[2].set_ylabel('Estacionalidad')
ax[3].plot(result.resid)
ax[3].set_ylabel('Residuos')
plt.tight_layout()
plt.show()

Depende del objetivo del análisis. Si el objetivo es hacer una predicción a largo plazo, puede ser más fácil estudiar una serie de tiempo con tendencia, ya que esto proporcionaría información sobre la dirección general en la que se mueve la serie a lo largo del tiempo. Por otro lado, si el objetivo es identificar patrones estacionales o ciclos repetitivos en la serie de tiempo, puede ser más fácil estudiar una serie de tiempo sin tendencia, ya que la tendencia puede ocultar estos patrones más sutiles. En última instancia, la elección depende del objetivo específico del análisis y de la naturaleza de la serie de tiempo en sí misma.

---
## 🧹 Fase 3 · Preparación de los datos

Hacemos la serie estacionaria (logaritmo, diferenciación) y verificamos con la prueba ADF.

📚 Lectura recomendada: [5]

In [ ]:
# Haciendo los datos estacionarios
df_testing = pd.DataFrame(df1.amount)
adf_check(df_testing.amount.dropna())
df_testing.plot()

In [ ]:
# Haciendo los datos estacionarios
df_testing = pd.DataFrame(np.log(df1.amount).diff().diff(12))
adf_check(df_testing.amount.dropna())
df_testing.plot()

El test de Dickey-Fuller Aumentado (ADF) es una prueba estadística que se utiliza para determinar si una serie temporal tiene una tendencia significativa. El valor de -5.225226609713747 es el valor del estadístico de prueba. Si el valor del estadístico es menor que el valor crítico, entonces se rechaza la hipótesis nula y se concluye que la serie temporal es estacionaria, lo que significa que no tiene una tendencia significativa.

En este caso, se podría inferir que el valor de -5.225226609713747 indica que la serie temporal es estacionaria y no tiene una tendencia significativa.


#### **Haciendo los datos estacionarios**



In [ ]:
df_testing

---
## 🤖 Fase 4 · Modelado — AR + MA a mano

Implementamos AR y MA desde cero y luego usamos `auto_arima` y `ARIMA` de `statsmodels`.

In [ ]:
# Fitting AR Model
best_RMSE=100000000000
best_p = -1

for i in range(1,21):
  [df_train,df_test,theta,intercept,RMSE] = AR(i,pd.DataFrame(df_testing.amount))
  if(RMSE<best_RMSE):
    best_RMSE = RMSE
    best_p = i

print(best_p)

In [ ]:
[df_train,df_test,theta,intercept,RMSE] = AR(best_p,pd.DataFrame(df_testing.amount))

In [ ]:
df_c = pd.concat([df_train,df_test])
df_c[['amount','Predicted_Values']].plot()

In [ ]:
res = pd.DataFrame()
res['Residuals'] = df_c.amount - df_c.Predicted_Values

In [ ]:
res

In [ ]:
res.plot(kind='kde')

In [ ]:
#Fitting MA on Residuals
best_RMSE=100000000000
best_q = -1

for i in range(1,19):
  [res_train,res_test,theta,intercept,RMSE] = MA(i,pd.DataFrame(res.Residuals))
  if(RMSE<best_RMSE):
    best_RMSE = RMSE
    best_q = i

print(best_q)

In [ ]:
res_c = pd.concat([res_train,res_test])
res_c

In [ ]:
df_c.Predicted_Values += res_c.Predicted_Values

In [ ]:
df_c[['amount','Predicted_Values']].plot()

In [ ]:
df_c.iloc[30:,:][['amount','Predicted_Values']].plot()

---
## 📊 Fase 5 · Evaluación

Partimos en entrenamiento/prueba **cronológicamente** y medimos el RMSE.

📚 Lectura recomendada: [3]

> 📌 En series de tiempo la partición entrenamiento/prueba es **cronológica**: se entrena con el pasado y se prueba con el futuro; nunca se mezclan al azar los periodos [3].

> ⚠️ **Nota:** parte del texto interpretativo de este cuaderno proviene de una versión en R; al ejecutar en Python los números exactos (AIC, coeficientes, RMSE) pueden variar ligeramente. Use las salidas de sus propias celdas para interpretar.

In [ ]:
df_testing=df_testing.dropna(axis=0)
train_size = int(len(df_testing) * 0.8)
train, test = df_testing[:train_size], df_testing[train_size:]

In [ ]:
model = auto_arima(train, trace=True, error_action='ignore', suppress_warnings=True)
print(model.summary())

In [ ]:
# Entrenar el modelo ARIMA
model = ARIMA(train, order=(2, 2, 1))
model_fit = model.fit()

In [ ]:
predictions = model_fit.forecast(steps=12)

In [ ]:
# Realizar la predicción
predictions = model_fit.forecast(steps=len(test))

In [ ]:
predictions

In [ ]:
test

In [ ]:
rmse = np.sqrt(mean_squared_error(test, predictions))
print("RMSE: {:.2f}".format(rmse))

In [ ]:
plt.plot(test, label='Valores Reales')
plt.plot(predictions, label='Predicciones')
plt.legend()
plt.show()

---
## 🧪 Refuerzo: error porcentual y línea base

Con `RMSE` solo no sabemos si el error es grande o pequeño respecto al tamaño de la serie. El **MAPE** lo expresa en %; y la línea base **ingenua** (repetir el último valor de entrenamiento) nos dice si el modelo aporta algo [3].

In [ ]:
import numpy as np
real = np.asarray(test, dtype=float).ravel()
est = np.asarray(predictions, dtype=float).ravel()
ingenuo = np.full_like(real, np.asarray(train, dtype=float).ravel()[-1])
print(f'MAPE modelo  : {np.mean(np.abs((real - est) / real)) * 100:.1f} %')
print(f'MAPE ingenuo : {np.mean(np.abs((real - ingenuo) / real)) * 100:.1f} %')

### ✍️ Ejercicios
1. ¿El modelo supera al pronóstico ingenuo? Si no, ¿qué cambiaría (órdenes, transformación)?
2. Repita con `np.log` aplicado a la serie y compare.

---
## ✅ Conclusiones y siguientes pasos

Construir AR y MA a mano ayuda a entender qué hace `auto_arima`: elegir cuántos rezagos del **pasado** (AR) y de los **errores** (MA) usar, tras volver estacionaria la serie [4]. Para **desplegar** el modelo: reentrenar periódicamente con datos nuevos, monitorear el RMSE y reportar intervalos de confianza junto al pronóstico.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [7], [9], [10].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3.ª ed.). OTexts. https://otexts.com/fpp3/

**[4]** Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5.ª ed.). Wiley.

**[5]** Dickey, D. A., & Fuller, W. A. (1979). Distribution of the estimators for autoregressive time series with a unit root. *Journal of the American Statistical Association, 74*(366a), 427–431. https://doi.org/10.1080/01621459.1979.10482531

**[6]** Cleveland, R. B., Cleveland, W. S., McRae, J. E., & Terpenning, I. (1990). STL: A seasonal-trend decomposition procedure based on loess. *Journal of Official Statistics, 6*(1), 3–73.

**[7]** Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011

**[8]** Smith, T. G., et al. (2017–). *pmdarima: ARIMA estimators for Python* [Software]. https://alkaline-ml.com/pmdarima

**[9]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[10]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55